In [0]:
%sh apt-get install -y osmium-tool

In [0]:
import subprocess
import json
import os

# 1. 경로 설정
input_pbf = "/dbfs/mnt/bronze/osm/raw/south-korea-latest.osm.pbf"
output_dir = "/dbfs/mnt/silver/seoul_districts"
config_file = "extract_config.json"

os.makedirs(output_dir, exist_ok=True)

# 2. Osmium Multi-extract용 설정 파일 생성
# 한 번의 실행으로 3개 구를 동시에 추출하도록 정의합니다.
extracts_config = {
    "extracts": [
        {
            "output": "gangnam.pbf",
            "bbox": [127.01, 37.45, 127.12, 37.54],
            "description": "Gangnam-gu extraction"
        },
        {
            "output": "seocho.pbf",
            "bbox": [126.97, 37.43, 127.09, 37.52],
            "description": "Seocho-gu extraction"
        },
        {
            "output": "songpa.pbf",
            "bbox": [127.07, 37.46, 127.18, 37.54],
            "description": "Songpa-gu extraction"
        }
    ],
    "directory": output_dir
}

with open(config_file, 'w') as f:
    json.dump(extracts_config, f)

# 3. Osmium 설치 (필요시)
subprocess.run(["apt-get", "update", "-y"], check=True)
subprocess.run(["apt-get", "install", "-y", "osmium-tool"], check=True)

# 4. Multi-extract 명령어 실행
print("🚀 Starting Multi-extract process...")
cmd = [
    "osmium", "extract",
    "--config", config_file,
    "--strategy", "complete_ways",
    input_pbf,
    "--overwrite"
]

result = subprocess.run(cmd, capture_output=True, text=True)

if result.returncode == 0:
    print("✅ Success! All 3 districts extracted in a single pass.")
    print(f"Files are located in: {output_dir}")
else:
    print(f"❌ Error: {result.stderr}")

In [0]:
# 1. 정보 설정 (본인 것으로 수정하세요)
storage_account_name = "3dtstorage"
container_name = "raw"
storage_key = "dpGtKlY+7its933u47+UYQSLCm3sJ2OYPqfrl6eRaH5jL6SNc023YLaHqAq68xZ0Jrqxm78SKsTt+AStMmpsnw=="

# 2. 마운트 지점 설정
mount_point = "/mnt/raw"

# 기존 마운트 해제 후 재마운트
if any(mount.mountPoint == mount_point for mount in dbutils.fs.mounts()):
    dbutils.fs.unmount(mount_point)

dbutils.fs.mount(
    source = f"wasbs://{container_name}@{storage_account_name}.blob.core.windows.net/",
    mount_point = mount_point,
    extra_configs = {f"fs.azure.account.key.{storage_account_name}.blob.core.windows.net": storage_key}
)
print("raw 마운트 완료!")

display(dbutils.fs.ls("/mnt/raw/seoul_api"))


/mnt/raw has been unmounted.
raw 마운트 완료!


path,name,size,modificationTime
dbfs:/mnt/raw/seoul_api/citydata_20260408_033105.json,citydata_20260408_033105.json,130809,1775619080000
dbfs:/mnt/raw/seoul_api/citydata_강남역_20260409_004644.json,citydata_강남역_20260409_004644.json,132957,1775695615000
dbfs:/mnt/raw/seoul_api/citydata_강남역_20260409_010305.json,citydata_강남역_20260409_010305.json,131752,1775696599000
dbfs:/mnt/raw/seoul_api/citydata_강남역_20260409_011303.json,citydata_강남역_20260409_011303.json,132085,1775697198000
dbfs:/mnt/raw/seoul_api/citydata_강남역_20260409_012302.json,citydata_강남역_20260409_012302.json,132119,1775697796000
dbfs:/mnt/raw/seoul_api/citydata_강남역_20260409_013303.json,citydata_강남역_20260409_013303.json,132163,1775698397000
dbfs:/mnt/raw/seoul_api/citydata_강남역_20260409_014303.json,citydata_강남역_20260409_014303.json,132262,1775699004000
dbfs:/mnt/raw/seoul_api/citydata_강남역_20260409_015303.json,citydata_강남역_20260409_015303.json,132226,1775699597000
dbfs:/mnt/raw/seoul_api/citydata_삼성역_20260409_004644.json,citydata_삼성역_20260409_004644.json,152,1775695615000
dbfs:/mnt/raw/seoul_api/citydata_삼성역_20260409_010305.json,citydata_삼성역_20260409_010305.json,152,1775696596000


In [0]:
# 마운트 해제 후 재마운트
dbutils.fs.unmount("/mnt/raw")
dbutils.fs.unmount("/mnt/silver")

storage_key = "dpGtKlY+7its933u47+UYQSLCm3sJ2OYPqfrl6eRaH5jL6SNc023YLaHqAq68xZ0Jrqxm78SKsTt+AStMmpsnw=="

# raw 재마운트
dbutils.fs.mount(
    source = "wasbs://raw@3dtstorage.blob.core.windows.net/",
    mount_point = "/mnt/raw",
    extra_configs = {"fs.azure.account.key.3dtstorage.blob.core.windows.net": storage_key}
)

# silver 재마운트
dbutils.fs.mount(
    source = "wasbs://silver@3dtstorage.blob.core.windows.net/",
    mount_point = "/mnt/silver",
    extra_configs = {"fs.azure.account.key.3dtstorage.blob.core.windows.net": storage_key}
)

print("raw, silver 재마운트 완료!")

/mnt/raw has been unmounted.
/mnt/silver has been unmounted.
raw, silver 재마운트 완료!


In [0]:
df_raw = spark.read.option("multiline", "true").json("/mnt/raw/seoul_api/citydata_20260408_033105.json")
display(df_raw)

---------------------------------------------------------------------------
ExecutionError                            Traceback (most recent call last)
File <command-4687067734452144>, line 21
     18     print("이미 마운트되어 있습니다.")
     20 # 4. 파일이 잘 보이는지 확인
---> 21 display(dbutils.fs.ls("/mnt/raw/seoul_api"))

File /databricks/python_shell/lib/dbruntime/dbutils.py:158, in prettify_exception_message.<locals>.f_with_exception_handling(*args, **kwargs)
    156 exc.__context__ = None
    157 exc.__cause__ = None
--> 158 raise exc

ExecutionError: An error occurred while calling o441.ls.
: shaded.databricks.org.apache.hadoop.fs.azure.AzureException: hadoop_azure_shaded.com.microsoft.azure.storage.StorageException: Server failed to authenticate the request. Make sure the value of Authorization header is formed correctly including the signature.
	at shaded.databricks.org.apache.hadoop.fs.azure.AzureNativeFileSystemStore.retrieveMetadata(AzureNativeFileSystemStore.java:2678)
	at shaded.databrick

In [0]:
dbutils.fs.mount(
    source = "wasbs://silver@3dtstorage.blob.core.windows.net/",
    mount_point = "/mnt/silver",
    extra_configs = {"fs.azure.account.key.3dtstorage.blob.core.windows.net": storage_key}
)
print("silver 마운트 완료!")

---------------------------------------------------------------------------
ExecutionError                            Traceback (most recent call last)
File <command-4687067734452151>, line 1
----> 1 dbutils.fs.mount(
      2     source = "wasbs://silver@3dtstorage.blob.core.windows.net/",
      3     mount_point = "/mnt/silver",
      4     extra_configs = {"fs.azure.account.key.3dtstorage.blob.core.windows.net": storage_key}
      5 )
      6 print("silver 마운트 완료!")

File /databricks/python_shell/lib/dbruntime/dbutils.py:158, in prettify_exception_message.<locals>.f_with_exception_handling(*args, **kwargs)
    156 exc.__context__ = None
    157 exc.__cause__ = None
--> 158 raise exc

ExecutionError: An error occurred while calling o445.mount.
: java.rmi.RemoteException: java.lang.IllegalArgumentException: requirement failed: Directory already mounted: /mnt/silver; nested exception is: 
	java.lang.IllegalArgumentException: requirement failed: Directory already mounted: /mnt/silver
	at